# Lab 04 - Janelamento Avançado e Watermarking
Neste laboratório, exploraremos como lidar com dados atrasados e janelas de tempo deslizantes (*Sliding Windows*).

**Objetivos:**
1. Implementar janelas deslizantes.
2. Utilizar Watermarking para gerenciar o estado e descartar dados tardios.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# 1. Definição do Schema e Path (Usando dataset de eventos nativo do Databricks)
inputPath = "/databricks-datasets/structured-streaming/events/"
jsonSchema = StructType([ 
    StructField("time", TimestampType(), True), 
    StructField("action", StringType(), True) 
])

# 2. Leitura do Stream com limitação de arquivos por gatilho para simular tempo real
df_streaming = (spark.readStream
                .schema(jsonSchema)
                .option("maxFilesPerTrigger", 1)
                .json(inputPath))

In [0]:
# 3. Aplicação de Watermarking e Janelamento Deslizante
# Definimos um limite de 10 minutos para dados atrasados (Watermark)
# Criamos uma janela de 10 minutos que desliza a cada 5 minutos
df_windowed = (df_streaming
               .withWatermark("time", "10 minutes")
               .groupBy(window(col("time"), "10 minutes", "5 minutes"), col("action"))
               .count())

In [0]:
# 4. Escrita do Stream para visualização em memória

# Parar eventuais queries ativas anteriores
for stream in spark.streams.active:
    stream.stop()

# Limpeza do diretório de checkpoint específico do Lab 04
checkpoint_path = "/Volumes/workspace/default/checkpoint/lab04_checkpoint"
dbutils.fs.rm(checkpoint_path, True)

query = (df_windowed.writeStream
         .format("memory")
         .queryName("contagem_janelada")
         .outputMode("complete")
         .option("checkpointLocation", checkpoint_path)
         .trigger(availableNow=True)
         .start())

In [0]:
# 5. Consulta dos resultados das janelas
display(spark.sql("SELECT window.start, window.end, action, count FROM contagem_janelada ORDER BY start DESC"))

In [0]:
# 6. Encerramento gracioso da query
print(f"Status da query antes de parar: {query.status}")
query.stop()
print("Query de streaming encerrada com sucesso.")